Fine-tune a RoBERTa-base model as a binary entity matching classifier.

This is the baseline model (Task 2). It is trained only on the original seed
training data without any augmentation, establishing a performance floor that
all augmentation strategies (Tasks 3–5) are compared against.

Architecture mirrors Ditto:
  - Model  : roberta-base
  - Input  : [CLS] left_entity [SEP] right_entity [SEP]
  - Output : binary classification (0 = non-match, 1 = match)
  - Loss   : cross-entropy


In [1]:
!unzip -q data.zip

In [2]:
import sys
from pathlib import Path
import json

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from dotenv import load_dotenv

load_dotenv()

from metrics import compute_metrics, full_report, confusion

In [3]:
ROOT = Path().cwd()
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

MODELS = ROOT / "experiments" / "models"
MODELS.mkdir(parents=True, exist_ok=True)

RESULTS = ROOT / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

Data loading

In [4]:
def load_pairs(path: Path) -> tuple[list[str], list[str], list[int]]:
    """Read a Ditto-format .txt file into (lefts, rights, labels)."""
    lefts, rights, labels = [], [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 3:
                continue
            lefts.append(parts[0])
            rights.append(parts[1])
            labels.append(int(parts[2]))
    return lefts, rights, labels

In [5]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256  # max tokens per pair



def build_hf_dataset(
    path: Path,
    tokenizer,
    max_length: int = MAX_LENGTH,
) -> Dataset:
    lefts, rights, labels = load_pairs(path)

    encodings = tokenizer(
        lefts,
        rights,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )
    encodings["labels"] = labels
    return Dataset.from_dict(encodings)

HuggingFace compute_metrics callback

In [6]:
def make_compute_metrics_fn():
    def _compute(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1).tolist()
        labels = labels.tolist()
        return compute_metrics(labels, preds)

    return _compute

Make weighted trainer

In [7]:
class WeightedTrainer(Trainer):
    """Trainer subclass that applies class weights to the cross-entropy loss."""

    def __init__(self, class_weights: torch.Tensor, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_class_weights(labels: list[int]) -> torch.Tensor:
    """Inverse-frequency weights: w_c = n_total / (n_classes * n_c)."""
    counts = [labels.count(0), labels.count(1)]
    n = len(labels)
    weights = [n / (2 * c) if c > 0 else 1.0 for c in counts]
    print(f"  Class weights: non-match={weights[0]:.3f}, match={weights[1]:.3f}")
    return torch.tensor(weights, dtype=torch.float)

Training

In [8]:
def train(
    dataset: str,
    epochs: int = 10,
    batch_size: int = 8,
    grad_accum: int = 4,
    lr: float = 5e-5,
    seed: int = 42,
    class_weight: str = "none",
    run_name: str = "baseline",
    train_file: Path | None = None,
):
    data_dir = PROCESSED / dataset
    if train_file is None:
        train_file = data_dir / "train.txt"
    valid_file = data_dir / "valid.txt"

    if not train_file.exists():
        sys.exit(
            f"[error] {train_file} not found. Run src/data_prep/preprocess.py first."
        )

    model_out = MODELS / f"{run_name}_{dataset}"
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Training {run_name} on {dataset} ===")
    print(f"  Model        : {MODEL_NAME}")
    print(f"  Train file   : {train_file}")
    print(f"  Epochs       : {epochs}")
    print(f"  Batch size   : {batch_size}  (grad_accum={grad_accum}, effective={batch_size*grad_accum})")
    print(f"  LR           : {lr}")
    print(f"  Class weight : {class_weight}")
    print(f"  Output dir   : {model_out}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    print("  Tokenizing training set...")
    train_ds = build_hf_dataset(train_file, tokenizer)

    eval_ds = None
    if valid_file.exists():
        print("  Tokenizing validation set...")
        eval_ds = build_hf_dataset(valid_file, tokenizer)

    training_args = TrainingArguments(
        output_dir=str(model_out),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch" if eval_ds else "no",
        save_strategy="epoch" if eval_ds else "epoch",
        load_best_model_at_end=True if eval_ds else False,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)] if eval_ds else []

    if class_weight == "balanced":
        _, _, train_labels = load_pairs(train_file)
        weights = compute_class_weights(train_labels)
        trainer = WeightedTrainer(
            class_weights=weights,
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            compute_metrics=make_compute_metrics_fn(),
            callbacks=callbacks,
        )
    else:
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            compute_metrics=make_compute_metrics_fn(),
            callbacks=callbacks,
        )

    trainer.train()

    # Save final best model + tokenizer
    trainer.save_model(str(model_out / "best"))
    tokenizer.save_pretrained(str(model_out / "best"))
    print(f"\n  Best model saved to {model_out / 'best'}")

# LLM Augmentation Training (llm_aug_cw on WDC, llm_aug on DBLP)
WDC-Product Training

In [9]:
train(dataset="wdc-products", class_weight="balanced", run_name="llm_aug_cw",
      train_file=Path("/content/data/processed/wdc-products/train_aug_llm.txt"))


=== Training llm_aug_cw on wdc-products ===
  Model        : roberta-base
  Train file   : /content/data/processed/wdc-products/train_aug_llm.txt
  Epochs       : 10
  Batch size   : 8  (grad_accum=4, effective=32)
  LR           : 5e-05
  Class weight : balanced
  Output dir   : /content/experiments/models/llm_aug_cw_wdc-products


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Best model saved to /content/experiments/models/llm_aug_cw_wdc-products/best


In [10]:
import os
import shutil

def delete_checkpoint_folders(path):
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path) and item.startswith("checkpoint"):
            shutil.rmtree(item_path)
            print(f"Deleted: {item_path}")

In [11]:
delete_checkpoint_folders("/content/experiments/models/llm_aug_cw_wdc-products")

Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-144
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-1152
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-432
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-720
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-288
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-1008
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-864
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-1296
Deleted: /content/experiments/models/llm_aug_cw_wdc-products/checkpoint-576


DBLP-Scholar Training

In [13]:
train(dataset="dblp-scholar", run_name="llm_aug",
      train_file=Path("/content/data/processed/dblp-scholar/train_aug_llm.txt"))


=== Training llm_aug on dblp-scholar ===
  Model        : roberta-base
  Train file   : /content/data/processed/dblp-scholar/train_aug_llm.txt
  Epochs       : 10
  Batch size   : 8  (grad_accum=4, effective=32)
  LR           : 5e-05
  Class weight : none
  Output dir   : /content/experiments/models/llm_aug_dblp-scholar


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Tokenizing training set...
  Tokenizing validation set...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.394115,0.157579,0.965447,0.887850,0.925024
2,0.356862,0.098500,0.913580,0.968224,0.940109
3,0.306030,0.109506,0.923146,0.965421,0.943810
4,0.185052,0.088168,0.922467,0.978505,0.949660
5,0.240290,0.087094,0.956357,0.942056,0.949153
6,0.107783,0.095339,0.951673,0.957009,0.954334
7,0.092024,0.126564,0.960915,0.942056,0.951392
8,0.078783,0.109160,0.952734,0.960748,0.956724
9,0.021607,0.124142,0.955827,0.950467,0.953140
10,0.011707,0.141722,0.958294,0.944860,0.951529


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Best model saved to /content/experiments/models/llm_aug_dblp-scholar/best


In [14]:
delete_checkpoint_folders("/content/experiments/models/llm_aug_dblp-scholar")

Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-1208
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-4832
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-5436
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-6040
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-3624
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-604
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-3020
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-4228
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-2416
Deleted: /content/experiments/models/llm_aug_dblp-scholar/checkpoint-1812


# Evaluation

In [16]:
def evaluate(dataset: str, split: str = "test",
             run_name: str = "baseline",
             save_preds: bool = False):

    model_dir = MODELS / f"{run_name}_{dataset}" / "best"
    data_file = PROCESSED / dataset / f"{split}.txt"

    if not model_dir.exists():
        sys.exit(
            f"[error] Model not found at {model_dir}. "
            "Run src/baseline/train_baseline.py first."
        )
    if not data_file.exists():
        sys.exit(
            f"[error] {data_file} not found. "
            "Run src/data_prep/preprocess.py first."
        )

    print(f"\n=== Evaluating {run_name} on {dataset} [{split}] ===")


    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))

    ds = build_hf_dataset(data_file, tokenizer)

    eval_args = TrainingArguments(
        output_dir=str(MODELS / "tmp_eval"),
        per_device_eval_batch_size=64,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=eval_args)

    output = trainer.predict(ds)
    logits = output.predictions
    preds = np.argmax(logits, axis=-1).tolist()

    exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    scores = (exp / exp.sum(axis=-1, keepdims=True))[:, 1].tolist()

    lefts, rights, labels = load_pairs(data_file)
    metrics = compute_metrics(labels, preds)
    cm = confusion(labels, preds)

    print(f"\n  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"\n{full_report(labels, preds)}")

    result = {
        "run": run_name,
        "dataset": dataset,
        "split": split,
        "model": str(model_dir),
        "metrics": metrics,
        "confusion_matrix": cm,
        "n_pairs": len(labels),
        "n_matches": sum(labels),
        "n_non_matches": len(labels) - sum(labels),
    }
    out_file = RESULTS / f"{run_name}_{dataset}_{split}.json"
    with open(out_file, "w") as f:
        json.dump(result, f, indent=2)
    print(f"\n  Results saved to {out_file}")

    if save_preds:
        preds_file = RESULTS / f"{run_name}_{dataset}_{split}_preds.jsonl"
        with open(preds_file, "w") as f:
            for left, right, true_label, pred_label, score in zip(lefts, rights, labels, preds, scores):
                f.write(json.dumps({
                    "left": left,
                    "right": right,
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "score": round(score, 6),
                }) + "\n")
        print(f"  Predictions saved to {preds_file}")

    return metrics

Evaluate LLM aug

In [17]:
llm_wdc_metrics = evaluate(dataset="wdc-products", split="test", run_name="llm_aug_cw", save_preds=True)


=== Evaluating llm_aug_cw on wdc-products [test] ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  Precision : 0.5099
  Recall    : 0.7720
  F1        : 0.6142

              precision    recall  f1-score   support

   non-match       0.97      0.91      0.94      4000
       match       0.51      0.77      0.61       500

    accuracy                           0.89      4500
   macro avg       0.74      0.84      0.78      4500
weighted avg       0.92      0.89      0.90      4500


  Results saved to /content/experiments/results/llm_aug_cw_wdc-products_test.json
  Predictions saved to /content/experiments/results/llm_aug_cw_wdc-products_test_preds.jsonl


In [18]:
llm_dblp_metrics = evaluate(dataset="dblp-scholar", split="test", run_name="llm_aug", save_preds=True)


=== Evaluating llm_aug on dblp-scholar [test] ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  Precision : 0.9498
  Recall    : 0.9542
  F1        : 0.9520

              precision    recall  f1-score   support

   non-match       0.99      0.99      0.99      4672
       match       0.95      0.95      0.95      1070

    accuracy                           0.98      5742
   macro avg       0.97      0.97      0.97      5742
weighted avg       0.98      0.98      0.98      5742


  Results saved to /content/experiments/results/llm_aug_dblp-scholar_test.json
  Predictions saved to /content/experiments/results/llm_aug_dblp-scholar_test_preds.jsonl


# Web Augmentation Training (web_aug_cw on WDC, web_aug on DBLP)



WDC-Product Training

In [19]:
train(dataset="wdc-products", class_weight="balanced", run_name="web_aug_cw",
      train_file=Path("/content/data/processed/wdc-products/train_aug_web.txt"))


=== Training web_aug_cw on wdc-products ===
  Model        : roberta-base
  Train file   : /content/data/processed/wdc-products/train_aug_web.txt
  Epochs       : 10
  Batch size   : 8  (grad_accum=4, effective=32)
  LR           : 5e-05
  Class weight : balanced
  Output dir   : /content/experiments/models/web_aug_cw_wdc-products


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Tokenizing training set...
  Tokenizing validation set...
  Class weights: non-match=0.767, match=1.437


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,2.261786,0.463278,0.455335,0.734000,0.562021
2,1.505773,0.380252,0.541920,0.892000,0.674225
3,1.341205,0.395218,0.736264,0.670000,0.701571
4,1.042618,0.491938,0.817919,0.566000,0.669031
5,0.742386,0.671921,0.785311,0.556000,0.651054
6,0.585905,0.544387,0.765864,0.700000,0.731452
7,0.565937,0.677645,0.822222,0.592000,0.688372
8,0.401224,0.807161,0.812183,0.640000,0.715884
9,0.273540,0.888665,0.796163,0.664000,0.724100


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Best model saved to /content/experiments/models/web_aug_cw_wdc-products/best


In [20]:
delete_checkpoint_folders("/content/experiments/models/web_aug_cw_wdc-products")

Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-728
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-208
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-624
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-520
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-832
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-936
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-104
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-416
Deleted: /content/experiments/models/web_aug_cw_wdc-products/checkpoint-312


DBLP-Scholar Training

In [22]:
train(dataset="dblp-scholar", run_name="web_aug",
      train_file=Path("/content/data/processed/dblp-scholar/train_aug_web.txt"))


=== Training web_aug on dblp-scholar ===
  Model        : roberta-base
  Train file   : /content/data/processed/dblp-scholar/train_aug_web.txt
  Epochs       : 10
  Batch size   : 8  (grad_accum=4, effective=32)
  LR           : 5e-05
  Class weight : none
  Output dir   : /content/experiments/models/web_aug_dblp-scholar


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Tokenizing training set...
  Tokenizing validation set...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.599079,0.150547,0.966492,0.862617,0.911605
2,0.651623,0.155868,0.868595,0.982243,0.921930
3,0.352614,0.105121,0.926457,0.965421,0.945538
4,0.325536,0.072315,0.927240,0.976636,0.951297
5,0.209311,0.100093,0.959203,0.944860,0.951977
6,0.164290,0.098792,0.957627,0.950467,0.954034
7,0.137555,0.105956,0.954503,0.960748,0.957615
8,0.041242,0.106633,0.959624,0.955140,0.957377
9,0.041036,0.133111,0.963740,0.943925,0.953730
10,0.030565,0.123979,0.954334,0.957009,0.955670


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Best model saved to /content/experiments/models/web_aug_dblp-scholar/best


In [23]:
delete_checkpoint_folders("/content/experiments/models/web_aug_dblp-scholar")

Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-1686
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-2810
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-5058
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-3934
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-3372
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-5620
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-562
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-2248
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-1124
Deleted: /content/experiments/models/web_aug_dblp-scholar/checkpoint-4496


Evaluate web aug

In [24]:
web_wdc_metrics = evaluate(dataset="wdc-products", split="test", run_name="web_aug_cw", save_preds=True)


=== Evaluating web_aug_cw on wdc-products [test] ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  Precision : 0.5963
  Recall    : 0.7180
  F1        : 0.6515

              precision    recall  f1-score   support

   non-match       0.96      0.94      0.95      4000
       match       0.60      0.72      0.65       500

    accuracy                           0.91      4500
   macro avg       0.78      0.83      0.80      4500
weighted avg       0.92      0.91      0.92      4500


  Results saved to /content/experiments/results/web_aug_cw_wdc-products_test.json
  Predictions saved to /content/experiments/results/web_aug_cw_wdc-products_test_preds.jsonl


In [25]:
web_dblp_metrics = evaluate(dataset="dblp-scholar", split="test", run_name="web_aug", save_preds=True)


=== Evaluating web_aug on dblp-scholar [test] ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  Precision : 0.9479
  Recall    : 0.9523
  F1        : 0.9501

              precision    recall  f1-score   support

   non-match       0.99      0.99      0.99      4672
       match       0.95      0.95      0.95      1070

    accuracy                           0.98      5742
   macro avg       0.97      0.97      0.97      5742
weighted avg       0.98      0.98      0.98      5742


  Results saved to /content/experiments/results/web_aug_dblp-scholar_test.json
  Predictions saved to /content/experiments/results/web_aug_dblp-scholar_test_preds.jsonl


In [26]:
!zip -r exp_results.zip /content/experiments

  adding: content/experiments/ (stored 0%)
  adding: content/experiments/models/ (stored 0%)
  adding: content/experiments/models/web_aug_dblp-scholar/ (stored 0%)
  adding: content/experiments/models/web_aug_dblp-scholar/best/ (stored 0%)
  adding: content/experiments/models/web_aug_dblp-scholar/best/model.safetensors (deflated 13%)
  adding: content/experiments/models/web_aug_dblp-scholar/best/tokenizer_config.json (deflated 51%)
  adding: content/experiments/models/web_aug_dblp-scholar/best/config.json (deflated 51%)
  adding: content/experiments/models/web_aug_dblp-scholar/best/tokenizer.json (deflated 82%)
  adding: content/experiments/models/web_aug_dblp-scholar/best/training_args.bin (deflated 53%)
  adding: content/experiments/models/web_aug_cw_wdc-products/ (stored 0%)
  adding: content/experiments/models/web_aug_cw_wdc-products/best/ (stored 0%)
  adding: content/experiments/models/web_aug_cw_wdc-products/best/model.safetensors (deflated 13%)
  adding: content/experiments/mod